# Cleaning Downloaded Data from avian-flu

Author: Alexander Maksiaev

Purpose: Clean downloaded data from avian-flu, rename sequences according to convention, de-duplicate from GISAID

In [1]:
# Housekeeping

import os
import glob 
import pandas as pd
import xml.etree.ElementTree as ET
import requests
import time
import numpy as np
import dateutil 
from datetime import datetime, timedelta
from collections import defaultdict 
import importlib
import utils  
importlib.reload(utils)
from utils import * 

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu"
# downloads = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
downloads = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
originals = downloads + "Andersen_Downloads/"
temp_files = downloads + "Andersen_Temp_Files/"
complete_files = downloads + "Andersen_Complete_Files/"

os.chdir(downloads)

In [2]:
# Read metadata

metadata_folder = originals + "avian-influenza/metadata/"
os.chdir(metadata_folder)

metadata = pd.read_csv("SraRunTable_automated.csv")

# print(len(metadata)) # 7397 rows

# Split date format to only check year
for date in metadata["Collection_Date"]:
    if "/" in date or date == "missing":
        metadata = metadata[metadata["Collection_Date"] != date]
metadata["Collection_Date"] = metadata["Collection_Date"].apply(lambda x: x.split("-")[0])
metadata["Collection_Date"] = metadata["Collection_Date"].apply(lambda x: int(x))

# Find only >= 2024 using run ID from metadata
metadata_new = metadata[metadata["Collection_Date"] >= 2024]
metadata_new["Collection_Date"] = metadata_new["Collection_Date"].astype(int) # Years are not floats

# Find only >= last date using Release Date from metadata (2025, 4, 1)
metadata_new["ReleaseDate"] = metadata_new["ReleaseDate"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y-%m-%d"))
metadata_new = metadata_new[metadata_new["ReleaseDate"] >= datetime(2024, 1, 1).strftime("%Y-%m-%d")]

print(len(metadata_new)) # 6053 rows between 1/1/2024 and 3/31/2025

C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\6\ipykernel_29312\80415036.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata_new["Collection_Date"] = metadata_new["Collection_Date"].astype(int) # Years are not floats


7044


C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\6\ipykernel_29312\80415036.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata_new["ReleaseDate"] = metadata_new["ReleaseDate"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y-%m-%d"))


### Naming convention ###
>A/[host]/[geo_loc_name]/[isolate]/[year]|[serotype: H5N1]|[collection_date]|[host_type]|[genotype: B3.13 or D1.1]

host_type is from manual animal reference

In metadata, we have: host, geo_loc_name, isolate, year

We need: geo_loc_name, collection_date, host_type, genotype

host = Host

geo_loc_name (primary) = geo_loc_name

geo_loc_name (secondary) = genbank_mapping.tsv > genbank_name

isolate = isolate

collection date (primary) = Collection_Date

collection date (secondary) = https://www.ncbi.nlm.nih.gov/genbank/ > BioSample (input: BioSample) > Nucleotide > [first result] > collection_date

serotype = serotype

host type = [from ref] 

genotype = [from genoflu] -- use genoflu_results.tsv

In [3]:
# # Get genotype from genoflu
# os.chdir(temp_files)
# output_tsv = pd.read_csv("output.tsv", delimiter="\t")

# b313_and_d11_only = output_tsv[(output_tsv["Genotype"] == "B3.13") | (output_tsv["Genotype"] == "D1.1")]
# b313_and_d11_only = b313_and_d11_only.rename(columns={"sample": "Run"})
# b313_and_d11_only = b313_and_d11_only.drop_duplicates(subset="Run", keep="last")
# # print(b313_and_d11_only)
# print(len(b313_and_d11_only)) # 5160 rows

# metadata_new = metadata_new.merge(b313_and_d11_only, on="Run", how="inner")

# print(len(metadata_new)) # 5160

In [4]:
# Get genotype from genoflu_results.tsv

output_tsv = pd.read_csv("genoflu_results.tsv", delimiter="\t")

b313_and_d11_only = output_tsv[output_tsv["Genotype"] == "D1.3"] #(output_tsv["Genotype"] == "B3.13") | (output_tsv["Genotype"] == "D1.1")]
b313_and_d11_only = b313_and_d11_only.rename(columns={"sample": "Run"})
b313_and_d11_only = b313_and_d11_only.drop_duplicates(subset="Run", keep="last")
# print(b313_and_d11_only)
print(len(b313_and_d11_only)) 

metadata_new = metadata_new.merge(b313_and_d11_only, on="Run", how="inner")

print(len(metadata_new)) 

278
278


In [5]:
print(len(metadata_new))

print(metadata_new)

278
             Run Assay Type  AvgSpotLen      Bases    BioProject  \
0    SRR32254505        WGS      144.35  123614013   PRJNA980729   
1    SRR32254524        WGS      145.94   89495943   PRJNA980729   
2    SRR32254525        WGS      145.31   85411542   PRJNA980729   
3    SRR32254527        WGS      145.54   90490423   PRJNA980729   
4    SRR32254528        WGS      145.69  106072236   PRJNA980729   
..           ...        ...         ...        ...           ...   
273  SRR32973821        WGS      148.38   90823853   PRJNA980729   
274  SRR32973832        WGS      148.69   84497018   PRJNA980729   
275  SRR32973833        WGS      148.33  104159552   PRJNA980729   
276  SRR33029801        WGS      145.93  295125424  PRJNA1207547   
277  SRR33029748        WGS      147.39  252799915   PRJNA980729   

        BioSample BioSampleModel      Bytes Center Name  Collection_Date  ...  \
0    SAMN46706291          Viral   44371660   USDA-NVSL             2025  ...   
1    SAMN46706273

In [6]:
# Get geolocation from genbank_mapping.tsv

genbank_mapping = pd.read_csv("genbank_mapping.tsv", delimiter="\t")
genbank_mapping["Run"] = genbank_mapping["sra_run"]
genbank_mapping = genbank_mapping.drop_duplicates(subset="Run", keep="first")
genbank_mapping["name_state"] = genbank_mapping["genbank_name"].apply(lambda x: x.split("/")[2])

# metadata_genbank = metadata_new.merge(genbank_mapping, on="Run", how="inner")
metadata_new["name_state"] = "unknown"
metadata_genbank = metadata_new

print(len(metadata_genbank))
display(metadata_genbank)

278


,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,retraction_detection_date_utc,date,File Name,Genotype,"Genotype List Used, >=98.0%",Genotype Sample Title List,Genotype Percent Match List,Genotype Mismatch List,Genotype Average Depth of Coverage List,name_state
0,SRR32254505,WGS,144.35,123614013,PRJNA980729,SAMN46706291,Viral,44371660,USDA-NVSL,2025,...,NaN,2025-04-08_15-56-55,SRR32254505.fa,D1.3,"PB2:am24, PB1:ea3, PA:am4, HA:ea3, NP:am13, NA...","am24:24-030039-001:PB2, ea3:22-013001-001:PB1,...","99.65%, 98.50%, 99.49%, 99.59%, 99.53%, 98.65%...","8, 34, 9, 7, 7, 19, 0, 9",Ran on FASTA - No Coverage Report,unknown
1,SRR32254524,WGS,145.94,89495943,PRJNA980729,SAMN46706273,Viral,32404657,USDA-NVSL,2025,...,NaN,2025-04-08_15-57-11,SRR32254524.fa,D1.3,"PB2:am24, PB1:ea3, PA:am4, HA:ea3, NP:am13, NA...","am24:24-030039-001:PB2, ea3:22-013001-001:PB1,...","99.61%, 98.28%, 98.19%, 99.41%, 99.67%, 98.72%...","9, 39, 39, 10, 5, 18, 0, 10",Ran on FASTA - No Coverage Report,unknown
2,SRR32254525,WGS,145.31,85411542,PRJNA980729,SAMN46706272,Viral,30633815,USDA-NVSL,2025,...,NaN,2025-04-08_15-57-12,SRR32254525.fa,D1.3,"PB2:am24, PB1:ea3, PA:am4, HA:ea3, NP:am13, NA...","am24:24-030039-001:PB2, ea3:22-013001-001:PB1,...","99.61%, 98.28%, 98.19%, 99.41%, 99.67%, 98.72%...","9, 39, 39, 10, 5, 18, 0, 10",Ran on FASTA - No Coverage Report,unknown
3,SRR32254527,WGS,145.54,90490423,PRJNA980729,SAMN46706271,Viral,32376254,USDA-NVSL,2025,...,NaN,2025-04-08_15-57-13,SRR32254527.fa,D1.3,"PB2:am24, PB1:ea3, PA:am4, HA:ea3, NP:am13, NA...","am24:24-030039-001:PB2, ea3:22-013001-001:PB1,...","99.61%, 99.25%, 99.49%, 99.53%, 99.60%, 98.65%...","9, 17, 11, 8, 6, 19, 0, 9",Ran on FASTA - No Coverage Report,unknown
4,SRR32254528,WGS,145.69,106072236,PRJNA980729,SAMN46706270,Viral,37963493,USDA-NVSL,2025,...,NaN,2025-04-08_15-57-14,SRR32254528.fa,D1.3,"PB2:am24, PB1:ea3, PA:am4, HA:ea3, NP:am13, NA...","am24:24-030039-001:PB2, ea3:22-013001-001:PB1,...","99.61%, 98.81%, 99.30%, 99.53%, 99.60%, 98.65%...","9, 27, 15, 8, 6, 19, 0, 9",Ran on FASTA - No Coverage Report,unknown
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
273,SRR32973821,WGS,148.38,90823853,PRJNA980729,SAMN47777552,Viral,32499008,USDA-NVSL,2025,...,NaN,2025-04-08_16-24-14,SRR32973821.fa,D1.3,"PB2:am24, PB1:ea3, PA:am4, HA:ea3, NP:am13, NA...","am24:24-030039-001:PB2, ea3:22-013001-001:PB1,...","99.61%, 99.30%, 99.44%, 99.59%, 99.60%, 98.65%...","9, 16, 12, 7, 6, 19, 0, 9",Ran on FASTA - No Coverage Report,unknown
274,SRR32973832,WGS,148.69,84497018,PRJNA980729,SAMN47777551,Viral,30111203,USDA-NVSL,2025,...,NaN,2025-04-08_16-24-23,SRR32973832.fa,D1.3,"PB2:am24, PB1:ea3, PA:am4, HA:ea3, NP:am13, NA...","am24:24-030039-001:PB2, ea3:22-013001-001:PB1,...","99.65%, 98.90%, 99.43%, 99.59%, 99.67%, 98.65%...","8, 25, 10, 7, 5, 19, 1, 9",Ran on FASTA - No Coverage Report,unknown
275,SRR32973833,WGS,148.33,104159552,PRJNA980729,SAMN47777550,Viral,36951162,USDA-NVSL,2025,...,NaN,2025-04-08_16-24-24,SRR32973833.fa,D1.3,"PB2:am24, PB1:ea3, PA:am4, HA:ea3, NP:am13, NA...","am24:24-030039-001:PB2, ea3:22-013001-001:PB1,...","99.61%, 98.90%, 98.42%, 99.59%, 99.60%, 98.65%...","9, 25, 34, 7, 6, 19, 0, 10",Ran on FASTA - No Coverage Report,unknown
276,SRR33029801,WGS,145.93,295125424,PRJNA1207547,SAMN47843945,Viral,102708374,USDA-NVSL,2025,...,NaN,2025-04-11_06-46-44,SRR33029801.fa,D1.3,"HA:ea3, NS:ea3, NA:ea3, PB1:ea3, MP:ea3, NP:am...","ea3:22-013001-001:HA, ea3:22-013001-001:NS, ea...","99.18%, 98.81%, 98.94%, 98.28%, 99.80%, 99.80%...","14, 10, 15, 39, 2, 3, 7, 8",Ran on FASTA - No Coverage Report,unknown


In [7]:
# Get collection date from GenBank eutils 

def search_collection_date(biosample):

    print(biosample)

    try:

        # Avoid spamming the server
        time.sleep(2)
    
        base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
        search_url = base_url + "esearch.fcgi?db=biosample&term=" + biosample +"&usehistory=y&api_key=2cbaf77ac9ec5ae7844ea350076ae6d56809"

        # Get Biosample ID from search_url
        output = requests.get(search_url)
        xml = output.content
        root = ET.fromstring(xml)
        sample_id = root.find("./IdList/Id").text

        biosample_url = base_url + "elink.fcgi?dbfrom=biosample&db=nuccore&id=" + sample_id + "&cmd=neighbor_history&api_key=2cbaf77ac9ec5ae7844ea350076ae6d56809"
        
        # Get Nucleotide ID from biosample_url
        output = requests.get(biosample_url)
        xml = output.content
        root = ET.fromstring(xml)
        query_key = root.find(".//QueryKey").text
        web_env = root.find(".//WebEnv").text

        nucleotide_url = base_url + "esummary.fcgi?db=nuccore&query_key=" + query_key + "&WebEnv=" + web_env + "&version=2.0&api_key=2cbaf77ac9ec5ae7844ea350076ae6d56809"

        output = requests.get(nucleotide_url) 
        xml = output.content
        root = ET.fromstring(xml)

        # Grab collection date at the end of the sub name
        collection_date = root.find(".//SubName").text.split("|")[-1]

        return collection_date
    
    except:
        print("Unable to find collection date.")

        if len(metadata_genbank[metadata_genbank["BioSample"] == biosample]["Collection_Date"]) > 0: # If a year exists
            collection_date = metadata_genbank[metadata_genbank["BioSample"] == biosample]["Collection_Date"].values[0]
        else:
            collection_date = float('nan') 

        return collection_date
    
metadata_genbank["Collection_Date_Specific"] = metadata_genbank["BioSample"].apply(search_collection_date)
# metadata_genbank["Collection_Date_Specific"] = metadata_genbank["years"]

SAMN46706291
Unable to find collection date.
SAMN46706273
Unable to find collection date.
SAMN46706272
Unable to find collection date.
SAMN46706271
Unable to find collection date.
SAMN46706270
Unable to find collection date.
SAMN46706269
Unable to find collection date.
SAMN46706268
Unable to find collection date.
SAMN46706267
Unable to find collection date.
SAMN46706266
Unable to find collection date.
SAMN46706265
Unable to find collection date.
SAMN46706264
Unable to find collection date.
SAMN46706263
Unable to find collection date.
SAMN46706254
Unable to find collection date.
SAMN46706253
Unable to find collection date.
SAMN46706252
Unable to find collection date.
SAMN46706251
Unable to find collection date.
SAMN46706250
Unable to find collection date.
SAMN46706249
Unable to find collection date.
SAMN46706248
Unable to find collection date.
SAMN46706195
Unable to find collection date.
SAMN46706194
Unable to find collection date.
SAMN46706209
Unable to find collection date.
SAMN467062

In [34]:
# Upload saved data
os.chdir(temp_files)
metadata_genbank = pd.read_csv("metadata_genbank.csv")

In [35]:
# # Save this so we don't have to do it again

# os.chdir(temp_files)
# metadata_genbank.to_csv("metadata_genbank.csv")

In [36]:

unique_animals_all = sort_animals_andersen(metadata_genbank)

# Flatten unique_animals
every_unique_animal = []
for animal in unique_animals_all:
    every_unique_animal.append(animal)

print(every_unique_animal)

unique_animals_set = list(set(every_unique_animal))
# animals_df = pd.DataFrame(columns=["avian", "cattle", "feline", "other_mammal", "human", "other"])
# animals_df["other"] = unique_animals_set # to sort

os.chdir(downloads)

animals_ref = pd.read_csv("animals_ref.csv")


# If animal not in ref1, put in ref2

common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and type(animal) == str:
            common_animals.append(animal)

print(common_animals)
print(len(common_animals))

different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print(different_animals)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# If longer, deal with that later

animals_df["new"] = (different_animals)

print(animals_df)

animals_df.to_csv("animals_ref_to_sort.csv")


['crow', 'goose', 'chicken', 'bald eagle', 'duck', 'great horned owl', 'mallard', 'sandhill crane', "cooper's hawk", 'red-tailed hawk', 'turkey', 'canada goose']
['crow', 'goose', 'chicken', 'bald eagle', 'duck', 'great horned owl', 'mallard', 'sandhill crane', "cooper's hawk", 'red-tailed hawk', 'turkey', 'canada goose']
12
[]
                      avian               cattle        feline   other_mammal  \
0          great_horned_owl            dairy_cow           cat         bobcat   
1              common_raven               cattle  domestic_cat    house_mouse   
2             cooper's_hawk  cattle milk product     feral_cat          skunk   
3              coopers_hawk                  NaN        feline         cougar   
4                   peafowl                  NaN  domestic-cat  geoffroys_cat   
..                      ...                  ...           ...            ...   
279           american crow                  NaN           NaN            NaN   
280  eurasian collared

In [37]:
# Get animals from animal reference
os.chdir(downloads)
animals_ref = pd.read_csv("animals_ref.csv")
fix_animals_andersen(metadata_genbank, animals_ref) # Get host type
metadata_genbank["years"] = metadata_genbank["Collection_Date"].apply(lambda x: str(x).split("-")[0]) # Get year only from collection date

In [38]:
for num, collection_date in enumerate(metadata_genbank["Collection_Date_Specific"]):
    if collection_date != collection_date: # If nan
        metadata_genbank.loc[num, "Collection_Date_Specific"] = metadata_genbank.loc[num, "years"]
    else: # If actual date
        if len(str(collection_date)) == 4: # If it's a year
            # print("caught")
            metadata_genbank.loc[num, "Collection_Date_Specific"] = collection_date
        else:
            parsed_date = dateutil.parser.parse(collection_date)
            date = parsed_date.strftime("%Y-%m-%d") # Make sure it doesn't default to today, if just a year
            metadata_genbank.loc[num, "Collection_Date_Specific"] = date

# Make names

names = ">A/" + metadata_genbank["Host"] + "/" + metadata_genbank["name_state"] + "/" + metadata_genbank["isolate"] + "/" + metadata_genbank["years"].apply(lambda x: str(x)) + "|H5N1|" + metadata_genbank["Collection_Date_Specific"].apply(lambda x: str(x)) + "|" + metadata_genbank["Host_Type"] + "|" + metadata_genbank["Genotype"]

metadata_genbank["Name"] = names

print(metadata_genbank["years"])

metadata_genbank.to_csv("metadata_genbank_named.csv")

# display(metadata_genbank)

0      2025
1      2025
2      2025
3      2025
4      2025
       ... 
273    2025
274    2025
275    2025
276    2025
277    2025
Name: years, Length: 278, dtype: object


In [39]:
# Make fasta files

fasta_folder = originals + "avian-influenza/fasta/"

os.chdir(fasta_folder)

pairs = []
fasta_files = {}

for genotype in ["D1.3"]:
    for segment in ["PB2", "PB1", "PA", "NS", "NP", "NA", "MP", "HA"]:
        pair = genotype + "_" + segment
        pairs.append(pair)

for pair in pairs:
    fasta_files[pair] = [] # List to hold fasta files

for run in metadata_genbank["Run"].values: # For each run 
    for dirpath, dirs, files in os.walk(fasta_folder): # Find the fasta file
        for file in files:
            file_name = os.path.join(dirpath, file) # Get file name
            # print(file_name)
            if run in file_name: # Note that there will be ~8 files total with that run name
                # Make a fasta file and put it in the list
                with open(file_name) as f:
                    lines = f.readlines()
                    sequence = lines[1] 
                    # Each run/segment pair has one sequence -- it's placed into a file with other run/segment pairs with the same segment and genotype
                    header = metadata_genbank[metadata_genbank["Run"] == run].loc[:, "Name"].values[0]
                    genotype = metadata_genbank[metadata_genbank["Run"] == run].loc[:, "Genotype"].values[0]
                    # print(header)
                    # print(genotype)
                    # break 
                    segment = file_name.split("_")[-2]
                    # Find the pair that corresponds to 
                    pair_name = genotype + "_" + segment
                    this_specific_fasta = []
                    for pair in pairs:
                        # print(pair)
                        # print(pair_name)
                        if pair_name == pair:
                            this_specific_fasta.append(header)
                            this_specific_fasta.append(sequence)
                            fasta_files[pair].append(this_specific_fasta)
                f.close()

In [41]:
# Create fasta files 
os.chdir(temp_files)

for pair in fasta_files.keys():
    output_path = temp_files + pair + ".fasta" 

    output_file = open(output_path, "w")
    for item in fasta_files[pair]:
        # for item in item:
        # item = fasta_files[pair]
        try:
            name = str(item[0].values[0]) # See if this is one we didn't have a collection date for
        except:
            name = str(item[0])
        print(name)
        # First is header, second is sequence
        # print(value)
        output_file.write(name + "\n")
        output_file.write(item[1])
    output_file.close()

>A/TURKEY/unknown/25-002414-001/2025|H5N1|2025|avian|D1.3
>A/CHICKEN/unknown/25-002299-002/2025|H5N1|2025|avian|D1.3
>A/CHICKEN/unknown/25-002299-001/2025|H5N1|2025|avian|D1.3
>A/TURKEY/unknown/25-002298-002/2025|H5N1|2025|avian|D1.3
>A/TURKEY/unknown/25-002298-001/2025|H5N1|2025|avian|D1.3
>A/TURKEY/unknown/25-002297-004/2025|H5N1|2025|avian|D1.3
>A/TURKEY/unknown/25-002297-003/2025|H5N1|2025|avian|D1.3
>A/TURKEY/unknown/25-002297-002/2025|H5N1|2025|avian|D1.3
>A/TURKEY/unknown/25-002297-001/2025|H5N1|2025|avian|D1.3
>A/TURKEY/unknown/25-002295-004/2025|H5N1|2025|avian|D1.3
>A/TURKEY/unknown/25-002295-003/2025|H5N1|2025|avian|D1.3
>A/TURKEY/unknown/25-002295-002/2025|H5N1|2025|avian|D1.3
>A/CHICKEN/unknown/25-002277-002/2025|H5N1|2025|avian|D1.3
>A/CHICKEN/unknown/25-002277-001/2025|H5N1|2025|avian|D1.3
>A/CHICKEN/unknown/25-002275-001/2025|H5N1|2025|avian|D1.3
>A/TURKEY/unknown/25-002274-002/2025|H5N1|2025|avian|D1.3
>A/TURKEY/unknown/25-002274-001/2025|H5N1|2025|avian|D1.3
>A/CHICKE

In [42]:
# De-duplication 

# Gisaid 

gisaid = downloads + "GISAID_Complete_Fasta_Files/" #04-01-2025--04-14-2025/"

os.chdir(gisaid)

def create_dataframes(directory):
    dfs_gisaid = defaultdict(list)
    for dirpath, dirs, files in os.walk(directory): # Find the fasta file
        for file in files:
            file_name = os.path.join(dirpath, file) # Get file name
            # print(file_name)
            gisaid_df = pd.DataFrame()
            with open(file_name) as f:
                lines = f.readlines()
                isolate_partial = []
                full_header = []
                sequence = []
                # Some lines start with 25_, others 25-. This shouldn't matter, but split on "_" first
                for num, line in enumerate(lines):
                    if line[0] == ">": # If it's a header
                        full_header.append(line)
                        full = line.split("/")[3] # Get the isolate
                        partial = full.split("_")[-1] # If 25_, get the last bit
                        digits = partial.split("-")
                        isolate = ""
                        other = ""
                        for d in digits:
                            # print(d)
                            if len(d) == 6 and d.isnumeric(): # If it's just digits and not one of those weird isolates
                                isolate = d + "-"
                            elif len(d) == 3 and d.isnumeric():
                                isolate = isolate + d
                            elif d.isnumeric() == False: # If it's a weird isolate
                                other = d + "-"
                            else: 
                                other = other + d
                        # Now add to list to check in Andersen files without doing wild for loops
                        if len(isolate) == 10: # If this is a correctly formatted isolate
                            # isolates.append(isolate)
                            # All headers are followed by sequences
                            isolate_partial.append(isolate)
                        else: # If this is some other isolate
                            isolate_partial.append(other)
                    elif line == "nan\n":
                        print(directory) # Some headers in the Andersen files don't exist 
                        isolate_partial.append(float('nan'))
                        full_header.append(line) # Sorry :/
                    else: # It's a sequence
                        sequence.append(line)
                    
                gisaid_df["isolate_partial"] = isolate_partial
                gisaid_df["full_header"] = full_header
                # print(len(sequence))
                gisaid_df["sequence"] = sequence
                dfs_gisaid[file_name.split("/")[-1][:-6]].append(gisaid_df)
    return dfs_gisaid

In [43]:
dfs_gisaid = create_dataframes(gisaid)
print(dfs_gisaid["B3.13_HA"][0])

    isolate_partial                                        full_header  \
0                P-  >A/dairy_cow/Idaho/W241290019-18-P/2024|H5N1|2...   
1                P-  >A/dairy_cow/Idaho/W241290019-19-P/2024|H5N1|2...   
2      W241220059-8  >A/dairy_cow/Idaho/W241220059-8/2024|H5N1|2024...   
3      W241220059-9  >A/dairy_cow/Idaho/W241220059-9/2024|H5N1|2024...   
4     W240870066-26  >A/dairy_cow/Idaho/W240870066-26/2024|H5N1|202...   
..              ...                                                ...   
202      000600-002  >A/dairy_cow/California/25_000600-002/2024|H5N...   
203      000590-002  >A/dairy_cow/California/25_000590-002/2024|H5N...   
204      005209-001  >A/dairy_cow/California/25_005209-001/2024|H5N...   
205      004920-005  >A/dairy_cow/California/25_004920-005/2024|H5N...   
206      004920-004  >A/dairy_cow/California/25_004920-004/2024|H5N...   

                                              sequence  
0    atggagaacatagtactacttcttgcaatagttagccttgttaaaa...

In [44]:
# Do the same with Andersen 

dfs_andersen = create_dataframes(temp_files)

In [46]:
print(dfs_andersen["D1.3_HA"][0])

    isolate_partial                                        full_header  \
0        002414-001  >A/TURKEY/unknown/25-002414-001/2025|H5N1|2025...   
1        002299-002  >A/CHICKEN/unknown/25-002299-002/2025|H5N1|202...   
2        002299-001  >A/CHICKEN/unknown/25-002299-001/2025|H5N1|202...   
3        002298-002  >A/TURKEY/unknown/25-002298-002/2025|H5N1|2025...   
4        002298-001  >A/TURKEY/unknown/25-002298-001/2025|H5N1|2025...   
..              ...                                                ...   
273      005210-003  >A/CHICKEN/unknown/25-005210-003/2025|H5N1|202...   
274      005210-002  >A/CHICKEN/unknown/25-005210-002/2025|H5N1|202...   
275      005210-001  >A/CHICKEN/unknown/25-005210-001/2025|H5N1|202...   
276      010298-001  >A/SANDHILL CRANE/unknown/25-010298-001/2025|H...   
277      010466-001  >A/CHICKEN/unknown/25-010466-001/2025|H5N1|202...   

                                              sequence  
0    ATGGAAAACATAGTACTTCTTCTTGCAATAATTAGCCTTGTTAAAA...

In [ ]:
# Merge dataframes and drop duplicates

full_dfs = defaultdict(list)
for key in dfs_andersen.keys():
    dataframes = dfs_andersen[key]
    for i, df in enumerate(dataframes):
        print(i)
        try:
            full_df = df.merge(dfs_gisaid[key][i], how="outer")
            # print(full_df)
            full_df = full_df.drop_duplicates(subset=["isolate_partial"])
            full_dfs[key].append(full_df)
        except:
            print("Failed to merge dataframes in ", key)



In [ ]:
# If none in one database, only use the other and drop duplicates

full_dfs = defaultdict(list)
for key in dfs_andersen.keys():
    print(key)
# for key in ["D1.3"]:
    dataframes = dfs_andersen[key]
    for i, df in enumerate(dataframes):
        print(i)
        try:
            # full_df = df.merge(dfs_gisaid[key][i], how="outer")
            # print(full_df)
            full_df = full_df.drop_duplicates(subset=["isolate_partial"])
            full_dfs[key].append(full_df)
        except:
            print("Failed to merge dataframes in ", key)

In [56]:
print(full_dfs["D1.3_HA"]) #[0])

[]


In [53]:
# Create fasta files 
os.chdir(complete_files)
for pair in full_dfs.keys():
    output_path = complete_files + pair + ".fasta" 

    output_file = open(output_path, "w")
    for item in full_dfs[pair]:
        # for item in item:
        # item = fasta_files[pair]
        for index, row in item.iterrows():
            name = item.loc[index, "full_header"]
            sequence = item.loc[index, "sequence"]
        # print(name)
        # First is header, second is sequence
        # print(value)
            output_file.write(name)
            output_file.write(sequence)
    output_file.close()